***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [5. 成像](5_0_introduction.ipynb)
    * 上一节： [5.4 脏图像与可见度权重](5_4_imaging_weights.ipynb)
    * 下一节： [5.x 延伸阅读与参考文献](5_x_further_reading_and_references.ipynb)

***


导入标准模块:

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    from IPython.display import HTML, Image
except ImportError:
    def HTML(*args, **kwargs):
        return None

    def Image(*args, **kwargs):
        return None

NOTEBOOK_DIR = Path("5_Imaging") if Path("5_Imaging").exists() else Path(".")
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
FIGURE_DIR = NOTEBOOK_DIR / "figures"
STYLE_PATH = NOTEBOOK_DIR.parent / "style" / "course.css"
TOGGLE_PATH = NOTEBOOK_DIR.parent / "style" / "code_toggle.html"

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))


def show_image(name, **kwargs):
    return Image(filename=str(FIGURE_DIR / name), **kwargs)


if STYLE_PATH.exists():
    try:
        HTML(STYLE_PATH.read_text(encoding="utf-8"))
    except Exception:
        pass

导入本节所需的专用模块:

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

import track_simulator

In [ ]:
if TOGGLE_PATH.exists():
    HTML(TOGGLE_PATH.read_text(encoding="utf-8"))

## 5.5 小角近似的失效与 w 项


到目前为止，我们一直借助重采样和快速傅里叶变换，在图像域与可见度域之间来回转换，并用下面这个简化的傅里叶关系来说明这一综合成像过程为什么成立：

\begin{equation}
 \begin{split}
     V(u,v) &= \int_\text{sky}{I(l,m)e^{-2\pi i/\lambda(\vec{b}\dot(\vec{s}-\vec{s}_0))}}dS\\
            &= \int\int{I(l,m)e^{-2\pi i/\lambda(ul+vm+w(\sqrt{1-l^2-m^2}-1))}}\frac{dldm}{\sqrt{1-l^2-m^2}}\\
            &\approx\int\int{I(l,m)e^{-2\pi i/\lambda(ul+vm)}}dldm\\
 \end{split}
\end{equation}

在这个表达式中，最后一步近似恰好就是标准的傅里叶变换，这也是我们前面一直用来成像的关系式。然而，更精确地把观测可见度与天球上的真实亮度分布联系起来的，其实并不是经典的二维傅里叶变换。

只有当 $n - 1 = \sqrt{1-l^2-m^2} - 1 \ll 1$（也就是成像区域足够小）以及/或者 $w \approx 0$（也就是阵列近似共面）时，这个近似才成立。这里的 $(n-1)$ 表示天球切平面近似与源在真实天球面上位置之间的投影高度差，如下图所示。


<img src="figures/orthogonal_projection_difference.png" alt="Smiley face" width="512">

*图：单位天球上的方向余弦，这里画出了 $l$ 与 $n$，其中 $n=\sqrt{1-l^2-m^2}$。如果投影极，也就是图像切点，恰好与参考相位中心重合，则有 $n_0 = 1$。源在切平面上的正交（SIN）投影与其在天球球面上的真实位置之间的误差可写为 $\epsilon=(n-n_0)=(\sqrt{1-l^2-m^2}-1)$。*


在窄视场且采样近似共面的条件下，使用 FFT 去构造天空的平面近似是合理的。本节将讨论：当视场变宽、基线不再共面时，会出现什么问题，以及这些问题应如何改正。


### 5.5.1 共面采样


假设有下面两个阵列：一个是一维的共面东-西阵列，另一个则是二维共面阵列，其中包含非东-西方向的基线。


In [ ]:
NO_ANTENNA = 4
NO_BASELINES = NO_ANTENNA * (NO_ANTENNA - 1) / 2 + NO_ANTENNA
CENTRE_CHANNEL = 1e9 / 299792458 #Wavelength of 1 GHz
#Create a perfectly planar array with both a perfectly East-West baseline and 2 2D baselines
ENU_2d = np.array([[5,0,0],
                [-5,0,0],
                [10,0,0],
                [0,23,0]]);
ENU_ew = np.array([[5,0,0],
                 [-5,0,0],
                 [10,0,0],
                 [0,0,0]]);

ARRAY_LATITUDE = 30 #Equator->North
ARRAY_LONGITUDE = 0 #Greenwitch->East, prime -> local meridian

In [ ]:
fig = plt.figure(figsize=(10, 5))
ax=fig.add_subplot(121)
ax.set_title("2D Array")
ax.plot(ENU_2d[:,0],ENU_2d[:,1],"ko")
ax.set_xlabel("East")
ax.set_ylabel("North")
ax.set_xlim(-30,30)
ax.set_ylim(-30,30)
ax=fig.add_subplot(122)
ax.set_title("East-west array")
ax.plot(ENU_ew[:,0],ENU_ew[:,1],"ko")
ax.set_xlabel("East")
ax.set_ylabel("North")
ax.set_xlim(-30,30)
ax.set_ylim(-30,30)
plt.show()

*图：两个假想平面阵列的 ENU 坐标。左图是二维阵列，右图是东-西阵列。*


与一维东-西阵列相比，二维阵列主要有两个优点：
1. 在较低赤纬下，它可以提供更好的 $uv$ 覆盖。
2. 当相位参考中心与基线方向正交时，干涉仪响应最大，因此保留多方向基线通常更有利。

特别是在较低仰角观测时，拥有不沿纯东-西方向排列的基线分量会更有价值。


In [ ]:
DECLINATION = 0
T_OBS = 12
T_INT = 1/60.0
uw_2hr_2d = track_simulator.sim_uv(0.0,DECLINATION,T_OBS,T_INT,ENU_2d,ARRAY_LATITUDE,False)/CENTRE_CHANNEL
uv_2hr_ew = track_simulator.sim_uv(0.0,DECLINATION,T_OBS,T_INT,ENU_ew,ARRAY_LATITUDE,False)/CENTRE_CHANNEL

fig = plt.figure(figsize=(10, 5))
ax=fig.add_subplot(121)
ax.set_title("2D Array")
ax.plot(uw_2hr_2d[:,0],uw_2hr_2d[:,1],'k.')
ax.set_xlabel("u")
ax.set_ylabel("v")
ax.set_xlim(-10,10)
ax.set_ylim(-10,10)
ax=fig.add_subplot(122)
ax.set_title("East-west Array")
ax.plot(uv_2hr_ew[:,0],uv_2hr_ew[:,1],'k.')
ax.set_xlabel("u")
ax.set_ylabel("v")
ax.set_xlim(-10,10)
ax.set_ylim(-10,10)
plt.show()

*图：在赤纬 $\delta=0$ 时，二维阵列与东-西阵列的 $uv$ 覆盖对比。*


不过，二维阵列也有一个重要缺点：即使所有天线都严格放在同一个物理平面上，随着地球自转，整段观测期间得到的测量也并不始终保持共面。相比之下，东-西向干涉仪的轨迹始终落在一个与地球赤道平行的平面上。下面给出的三维 $uvw$ 轨迹及其投影就展示了这一点。当然，如果观测时间非常短，也就是所谓的“快照”观测，那么地球转过的角度足够小，此时二维阵列的测量也可以近似看作共面。


In [ ]:
DECLINATION = 45
T_INT = 1/60.0
T_OBS = 12
uvw_2d = track_simulator.sim_uv(0.0,DECLINATION,T_OBS,T_INT,ENU_2d,ARRAY_LATITUDE,False)/CENTRE_CHANNEL
uvw_ew = track_simulator.sim_uv(0.0,DECLINATION,T_OBS,T_INT,ENU_ew,ARRAY_LATITUDE,False)/CENTRE_CHANNEL
fig=plt.figure(figsize=(10, 5))
ax=fig.add_subplot(121,projection='3d')
ax.set_title("2D Array")
ax.view_init(elev=10, azim=160)
ax.plot(uvw_2d[:,0],uvw_2d[:,1],uvw_2d[:,2],'k.')
ax.set_xlabel("u")
ax.set_ylabel("v")
ax.set_zlabel("w")
ax=fig.add_subplot(122,projection='3d')
ax.set_title("East-west array")
ax.view_init(elev=10, azim=160)
ax.plot(uvw_ew[:,0],uvw_ew[:,1],uvw_ew[:,2],'k.')
ax.set_xlabel("u")
ax.set_ylabel("v")
ax.set_zlabel("w")
plt.show()

fig = plt.figure(figsize=(10, 5))
ax=fig.add_subplot(121)
ax.set_title("2D Array")
ax.plot(uvw_2d[:,0],uvw_2d[:,1],'k.')
ax.set_xlabel("u")
ax.set_ylabel("v")
ax=fig.add_subplot(122)
ax.set_title("East-west array")
ax.plot(uvw_ew[:,0],uvw_ew[:,1],'k.')
ax.set_xlabel("u")
ax.set_ylabel("v")
plt.show()

*图：二维干涉仪与东-西向干涉仪的 $u,v,w$ 轨迹，以及它们在 $(u,v,w=0)$ 平面上的投影。*


当测量只沿着单一平面采样时，例如东-西向干涉仪，就可以把所有 $w$ 都写成同一组 $u$ 和 $v$ 的线性组合：$w = \alpha u + \beta v$。这虽然会让 $uv$ 坐标产生轻微畸变，但天空与测量之间的关系仍然可以保持为一个有效的二维傅里叶变换：

\begin{equation}
    \begin{split}
        V(u,v,w) &= \int\int{I(l,m)e^{-2\pi i/\lambda(ul' + vm')}\frac{dldm}{\sqrt{1-l^2-m^2}}}\\
        l' &= l + \alpha(\sqrt{1-l^2-m^2} - 1)\\
        m' &= m + \beta(\sqrt{1-l^2-m^2} - 1)\\
    \end{split}
\end{equation}


### 5.5.2 非共面采样

对于二维阵列，就不能这么简单处理了。$w$ 与 $u,v$ 之间并不存在固定不变的关系，而是同时依赖于随时间变化的天顶角和视差角。只有在瞬时观测，并且阵列本身又足够接近平面时，$uv$ 覆盖才可以近似认为是共面的。

如果对二维阵列进行宽视场成像时仍然忽略 $w(n-1)$ 这一项，并继续使用平面近似，就会引入方向依赖误差。这种相位误差与天线之间的高度差有关，下图中的倾斜干涉仪示意正好说明了这一点。


In [ ]:
show_image("tilted_interferometer.png")

*图：源矢量在共面干涉仪与倾斜干涉仪基线上的投影并不相同。对共面基线而言，沿着视线 $\vec{s}$ 测得的相位为 $\theta = \frac{-2\pi i}{\lambda}(ul + vm)$；而对倾斜基线，同一方向上的相位则变为 $\theta_\text{tilt} = \frac{-2\pi i}{\lambda}[ul + vm + w(n-1)]$。基线越长、源距离相位中心越远，这种传播时延误差就越显著。*


需要特别注意的是，这一额外相位项本质上纯粹来源于几何效应。对非共面测量而言，如果只是人为插入一个时延去修正 $\Delta w$，那也只能对某一条视线方向有效。换句话说，这种修正本质上只是改变了干涉仪的相位中心，而不是从根本上消除了宽视场误差。


类似于使用 $\sqrt{1+x} \approx 1+ \frac{x}{2}$ 的小角近似，我们可以对这种相位误差的影响形成直观理解。可以证明：

\begin{equation}
    V(u,v,w) \approx {\int\int{I(l,m)(e^{2\pi i/\lambda wl^2/2}e^{2\pi i /\lambda wm^2/2})e^{-2\pi i /\lambda(ul+vm)}\frac{dldm}{n}}}
\end{equation}

由于 $w$ 可以重写为 $u,v$ 以及随时间变化的高度角、方位角的复杂函数，因此源的位置会随着时间和基线不同而发生变化。这种相对位置偏移大致会随着源在 $l,m$ 上离相位中心的距离呈二次增长。下面几幅图展示的正是这种长时间宽视场观测导致的源拖尾现象，先是真实的 JVLA 观测，再是 MeerKAT 的模拟观测。


In [ ]:
show_image("vla_uncorrected.png")

*图：JVLA-D 对超新星遗迹 G55.7+3.4 的 8 小时观测图像，未做改正。注意点源周围明显的椭圆形拖尾。*


In [ ]:
show_image("vla_wproj.png")

*图：JVLA-D 对超新星遗迹 G55.7+3.4 的 8 小时观测图像，经过 W-projection 改正。*


In [ ]:
FIELD_SIZE_DEG = 10.0
WIDE_N = 96
field_axis_deg = np.linspace(-FIELD_SIZE_DEG / 2, FIELD_SIZE_DEG / 2, WIDE_N)
field_axis_rad = np.deg2rad(field_axis_deg)
ll_rad, mm_rad = np.meshgrid(field_axis_rad, field_axis_rad)
inside_sky = ll_rad ** 2 + mm_rad ** 2 < 0.95 ** 2
n_term = np.sqrt(np.clip(1.0 - ll_rad ** 2 - mm_rad ** 2, 1e-8, None))

wide_sky = (
    1.00 * np.exp(-((ll_rad + np.deg2rad(0.5)) ** 2 + (mm_rad - np.deg2rad(0.4)) ** 2) / (2 * np.deg2rad(0.20) ** 2))
    + 0.40 * np.exp(-((ll_rad - np.deg2rad(1.8)) ** 2 + (mm_rad + np.deg2rad(1.5)) ** 2) / (2 * np.deg2rad(0.35) ** 2))
)
point_sources_deg = [(3.2, -2.5, 0.70), (-3.5, 2.8, 0.42), (2.1, 3.0, 0.25)]
for l0_deg, m0_deg, flux in point_sources_deg:
    ix = np.argmin(np.abs(field_axis_deg - l0_deg))
    iy = np.argmin(np.abs(field_axis_deg - m0_deg))
    wide_sky[iy, ix] += flux
wide_sky[~inside_sky] = 0.0

w_planes = np.array([-1500.0, -900.0, -450.0, 0.0, 450.0, 900.0, 1500.0])
representative_w = 1200.0
phase_screen = np.angle(np.exp(-2j * np.pi * representative_w * (n_term - 1.0)))

sky_over_n = wide_sky / n_term
vis_planes = []
image_planes = []
for w_value in w_planes:
    sky_with_w = sky_over_n * np.exp(-2j * np.pi * w_value * (n_term - 1.0))
    vis_plane = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(sky_with_w)))
    vis_planes.append(vis_plane)
    image_planes.append(np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(vis_plane))))

vis_planes = np.array(vis_planes)
image_planes = np.array(image_planes)
uncorrected_image = np.real(np.mean(image_planes, axis=0))
wide_extent = [field_axis_deg[0], field_axis_deg[-1], field_axis_deg[0], field_axis_deg[-1]]

print(f"Synthetic w planes used in the demo: {len(w_planes)}")
print(f"w range: {w_planes.min():.0f} to {w_planes.max():.0f} wavelengths")
print(f"Representative |w| for the phase screen: {representative_w:.0f} wavelengths")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(phase_screen, origin='lower', extent=wide_extent, cmap='twilight')
axes[0].set_title('Image-domain phase screen')
axes[0].set_xlabel('l (deg)')
axes[0].set_ylabel('m (deg)')

axes[1].imshow(uncorrected_image, origin='lower', extent=wide_extent, cmap='viridis')
axes[1].set_title('Image formed without w correction')
axes[1].set_xlabel('l (deg)')
axes[1].set_ylabel('m (deg)')
plt.tight_layout()

*图：左图给出了某个代表性 $w$ 值对应的像平面相位屏 $\phi_w(l,m)=-2\pi w(n-1)$，可以看到其远离相位中心时迅速弯曲；右图则展示了把多层非零 $w$ 可见度简单平均后、仍按 $w=0$ 平面近似成图得到的结果。离相位中心更远的源会出现明显展宽、拖尾和振铃。*

In [ ]:
corrected_planes = []
for image_plane, w_value in zip(image_planes, w_planes):
    corrected_planes.append(np.real(image_plane * np.exp(2j * np.pi * w_value * (n_term - 1.0)) * n_term))
corrected_image = np.mean(corrected_planes, axis=0)

residual_uncorrected = uncorrected_image - wide_sky
residual_corrected = corrected_image - wide_sky
residual_scale = max(np.max(np.abs(residual_uncorrected)), np.max(np.abs(residual_corrected)))

off_axis_mask = (
    (ll_rad - np.deg2rad(3.2)) ** 2 + (mm_rad + np.deg2rad(2.5)) ** 2
) < np.deg2rad(0.45) ** 2

peak_true = np.max(wide_sky[off_axis_mask])
peak_uncorrected = np.max(uncorrected_image[off_axis_mask])
peak_corrected = np.max(corrected_image[off_axis_mask])

print(f"Off-axis source peak (true sky)          = {peak_true:.3f}")
print(f"Off-axis source peak (without correction)= {peak_uncorrected:.3f}")
print(f"Off-axis source peak (w-corrected)       = {peak_corrected:.3f}")
print(f"Global RMS residual without correction   = {np.sqrt(np.mean(residual_uncorrected ** 2)):.4f}")
print(f"Global RMS residual with correction      = {np.sqrt(np.mean(residual_corrected ** 2)):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].imshow(wide_sky, origin='lower', extent=wide_extent, cmap='viridis')
axes[0].set_title('Model sky')
axes[0].set_xlabel('l (deg)')
axes[0].set_ylabel('m (deg)')

axes[1].imshow(corrected_image, origin='lower', extent=wide_extent, cmap='viridis')
axes[1].set_title('Image after explicit w correction')
axes[1].set_xlabel('l (deg)')
axes[1].set_ylabel('m (deg)')

axes[2].imshow(residual_corrected, origin='lower', extent=wide_extent, cmap='coolwarm', vmin=-residual_scale, vmax=residual_scale)
axes[2].set_title('Residual after w correction')
axes[2].set_xlabel('l (deg)')
axes[2].set_ylabel('m (deg)')
plt.tight_layout()

*图：左图是同一模拟天空模型；中图是在每个 $w$ 层上显式乘回对应相位屏后，再把图层合成的结果；右图显示校正后的残差。由于这个实验只保留几何宽视场效应而不引入额外稀疏采样误差，因此可以更清楚地看到：一旦显式处理 $w(n-1)$，离轴源的形状与峰值都会明显恢复。*

### 5.5.3 非共面基线效应的改正


In [ ]:
LAMBDA_1GHZ = 299792458 / 1e9
example_longest_baseline_m = np.max(np.abs(w_planes)) * LAMBDA_1GHZ
example_dish_diameter_m = 13.5
quality_factor = 0.01
field_diameter_rad = np.deg2rad(FIELD_SIZE_DEG)
field_radius_rad = field_diameter_rad / 2.0
n_edge = np.sqrt(max(1.0 - 2.0 * field_radius_rad ** 2, 1e-6))
max_w = np.max(np.abs(w_planes))

n_facets = 2.0 * example_longest_baseline_m * LAMBDA_1GHZ / (quality_factor * example_dish_diameter_m ** 2)
n_planes = 2.0 * np.pi * max_w * abs(n_edge - 1.0) / quality_factor
w_support = 4.0 * np.pi * max_w * field_diameter_rad ** 2 / np.sqrt(2.0 - field_diameter_rad ** 2)

print(f"Illustrative longest baseline            = {example_longest_baseline_m:.1f} m")
print(f"Illustrative maximum |w|                 = {max_w:.1f} wavelengths")
print(f"Estimated facet count (xi={quality_factor}) = {n_facets:.1f}")
print(f"Estimated number of w planes            = {n_planes:.1f}")
print(f"Approximate w-kernel support            = {w_support:.1f} uv pixels")

在重采样和二维傅里叶变换时，如果忽略 $w(n-1)$，就会引入额外时延误差。针对这些误差，常见的改正思路包括：

**全三维变换：** 类似二维 DFT，也可以直接在 $l,m,n$ 组成的三维立方体中逐点计算傅里叶变换，天空信号位于这个立方体内的单位球面上。Perley 在 [<cite data-cite='taylor1999synthesis'>Synthesis Imaging in Radio Astronomy II</cite> &#10548;](http://adsabs.harvard.edu/full/1999ASPC..180..383P) 中详细推导了这种方法，但它在计算量和内存需求上通常都过于昂贵，因此很少实际采用。

**快照成像：** 如前所述，只要观测足够短，并且阵列物理布局近似平面，那么每次短时观测得到的可见度就可以视作共面。不过，由于每次快照对应的 $l,m$ 坐标会有细微扭曲，因此在把多张快照图像平均成一幅总图之前，仍需先把它们插值到同一坐标网格上。

**多面体成像（facet imaging）：** 这种方法的目标，是通过把 $(n-1)$ 压低到足够接近 0，从而重新满足二维傅里叶反演所需的窄视场假设。经典做法是把天球切分成许多较小的切平面图像，也就是把球面近似为一个多面体。

正切多面体成像法的实现思路比较直接：先把可见度相位旋转到每个小分块的图像中心 $(l_i,m_i)$，再让每个分块的几何方向一并旋转，使其在该点处与天球面相切。由于傅里叶变换保持旋转关系，因此可以通过旋转测量的 $u,v,w$ 坐标，来模拟干涉仪若指向新的相位中心 $(\alpha_i,\delta_i)$ 时会得到的轨迹。令 $(l_\Delta,m_\Delta,n_\Delta) = (l_i-l_0,m_i-m_0,n_i-n_0)$，则有：

\begin{equation}
  \begin{split}
    V(u,v,w)&\approx\int{\int{B(l-l_i,m-m_i,n-n_i)e^{-2{\pi}i[u(l-l_i)+v(m-m_i)+w(n-n_i)]}\frac{dldm}{n}}}\\
    &\approx\int{\int{B(l-l_i,m-m_i,n-n_i)e^{-2{\pi}i[u(l-l_0-l_\Delta)+v(m-m_0-m_\Delta)+w(n-n_0-n_\Delta)]}\frac{dldm}{n}}}\\
    &\approx\left[\int{\int{B(l-l_0,m-m_0,n-n_0)e^{-2{\pi}i[u(l-l_0)+v(m-m_0)+w(n-n_0)]}\frac{dldm}{n}}}\right]e^{2{\pi}i[ul_\Delta,vm_\Delta,wn_\Delta]}\\
  \end{split}
\end{equation}

如果只旋转相位中心，而不同时旋转分块几何，那么离原始相位中心越远的分块，其有效视场就必须越小，才能把投影误差控制在相近水平，这会显著增加计算负担。相反，如果让这些分块一起旋转，构成围绕天球面的“多面体”，那么所有分块就可以保持相同大小。


In [ ]:
show_image("coplanar-faceting.png")

In [ ]:
show_image("non-coplanar-faceting.png")

*图：如果连同分块几何一起旋转，那么各个分块就可以保持相同大小。*


把某个分块的相位中心移到 $(l_i,m_i)$ 之后，为了使该分块在该点处与天球面相切，还需要在可见度完成相位平移之后，利用下面的旋转矩阵来计算新的 $u',v',w'$ 坐标：

\begin{equation}
 \left[\begin{array}{c}
	u'\\
	v'\\
	w'\\
       \end{array} \right] = R(\alpha_i,\delta_i)R^{T}(\alpha_0,\delta_0)\left[\begin{array}{c}
						u\\
						v \\
						w \\
					      \end{array}
					\right]
\end{equation}

\begin{equation}
 R(\alpha,\delta) =
 \left[\begin{array}{c c c}
     -\sin{\alpha} 			& \cos{\alpha}		& 0 \\
     -\sin{\delta}\cos{\alpha} 	& -\sin{\delta}\sin{\alpha}	& \cos{\delta}\\
     \cos{\delta}\cos{\alpha} 	& \cos{\delta}\sin{\alpha}	& \sin{\delta}\\
    \end{array}\right]   
\end{equation}

满足采样准则时所需分块数量，可粗略估计为：

\begin{equation}
       N_\text{facets} = \frac{2L\lambda}{{\xi}D^2}, \; \xi\ll{1}
\end{equation}

其中，$L$ 是最长基线长度，$D$ 是天线口径，$\xi$ 是质量控制因子，它决定了允许分块图像偏离天球面的最大程度。

**W-projection：** W-projection 的基本思想，是把所有非共面测量都重新关联到 $w=0$ 的测量平面上，从而把相位中的 $w$ 项吸收到卷积核里。利用卷积定理，可得到 $V(u,v,w)$ 与 $V(u,v,0)$ 的关系：

\begin{equation}
       \begin{split}
           V &= \int\int{I(l,m)e^{-2\pi i[ul+vm]}e^{-2\pi i[w(n-1)]}\frac{dldm}{n}}\\
           V &= \int\int{I(l,m)e^{-2\pi i[ul+vm]}\mathcal{w}_w(l,m)\frac{dldm}{n}}\\
           V(u,v,w) &= V(u,v,w=0)\circ\mathcal{W}_w(u,v)\\
       \end{split}
\end{equation}

这意味着，在重采样时可以预先准备一组与不同 $w$ 值对应的滤波器，然后针对每条观测可见度，选取合适的滤波器与之卷积，把它映射到统一的 $uv$ 平面上。和窄视场成像一样，这些 $w$ 滤波器通常还要与抗混叠滤波器联合使用。

这些 $\mathcal{W}_w$ 滤波器的支持宽度与图像尺寸有关，可由下式估计：

\begin{equation}
       W_\text{sup} = \frac{4\pi w_\text{max}}{\lambda} \frac{D_\text{im}^2}{\sqrt{2-D_\text{im}^2}}
\end{equation}

其中 $D_\text{im}$ 表示图像大小。这也说明该方法的计算复杂度会随着图像尺寸增加而上升。

**W-stacking：** 与 W-projection 相对，W-stacking 是一种图像域方法，它把与 $w$ 有关的相位项直接乘到图像层上：

1. 在重采样过程中，把可见度按离散化后的不同 $w$ 值分配到多个网格上。
2. 对每一层网格分别做逆傅里叶变换。
3. 逐层乘以对应的 $\mathcal{w}_w(l,m)$。
4. 将全部图层平均合成为一张最终图像。

所需离散 $w$ 滤波器数量（用于 W-projection）或图像层数（用于 W-stacking）可由下式估算：

\begin{equation}
        N_\text{planes} = \frac{2\pi w_\text{max}(n_\text{edge}-1)}{\lambda_\text{min}\xi}, \; \xi\ll{1}
\end{equation}

$w$ 分层越细，也就是质量控制因子 $\xi$ 设得越严格，相位改正通常就越准确。


要点：当成像视场达到数度以上，尤其是所用阵列还包含长基线时，成像过程中通常必须启用宽视场改正。具体如何开启，应查阅所用成像工具的相关文档。


***

* 下一节： [5.x 延伸阅读与参考文献](5_x_further_reading_and_references.ipynb)
